# Introductory Statistics – Part 5: Two-Sample Inference

Welcome to Part 5. We extend hypothesis testing to situations involving **two groups**, which arise constantly in practice (e.g., treatment vs control, before vs after, men vs women).

**By the end of this notebook you will be able to:**

- Choose between and apply a **Welch** or **pooled-variance** two-sample $t$-test
- Perform a **paired $t$-test** for matched observations
- Conduct a **two-proportion $z$-test**
- Build and test a **contingency table** with the chi-square test of independence
- Apply the **$F$-test** to compare two variances

**Topics covered:**

16. Comparing two independent means (pooled vs Welch)  
17. Comparing paired means  
18. Comparing two population proportions  
19. Contingency tables and the chi-square test of independence  
20. $F$-test for comparing two variances

> **Prerequisite:** Part 4 – Power, p-values, and the t-Distribution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, ttest_rel, chi2_contingency, f, norm

sns.set(style="whitegrid")
np.random.seed(42)

---
## 16. Comparing Two Independent Means

When two groups are sampled **independently**, we compare their means using a two-sample $t$-test.

### Two versions

| Test | Assumption | Test statistic |
|---|---|---|
| **Pooled-variance** | Equal population variances ($\sigma_1^2 = \sigma_2^2$) | $t = \dfrac{\bar{x}_1 - \bar{x}_2}{s_p\sqrt{1/n_1 + 1/n_2}}$, $df = n_1 + n_2 - 2$ |
| **Welch's** | Unequal variances allowed | $t = \dfrac{\bar{x}_1 - \bar{x}_2}{\sqrt{s_1^2/n_1 + s_2^2/n_2}}$, $df$ approximated |

**Recommendation:** use Welch's test by default — equal variances are rarely guaranteed, and Welch's performs well in both cases.

In [ ]:
# Simulate two independent groups with slightly different means and spreads
np.random.seed(42)
group1 = np.random.normal(loc=70, scale=10, size=40)
group2 = np.random.normal(loc=75, scale=12, size=45)

# Welch's t-test (equal_var=False — the default, and recommended)
t_w, p_w = ttest_ind(group1, group2, equal_var=False)

# Pooled-variance t-test (equal_var=True)
t_p, p_p = ttest_ind(group1, group2, equal_var=True)

print("Group 1: n=40, μ≈70, σ≈10")
print("Group 2: n=45, μ≈75, σ≈12")
print()
print(f"Welch's t-test:          t = {t_w:.3f},  p = {p_w:.4f}")
print(f"Pooled-variance t-test:  t = {t_p:.3f},  p = {p_p:.4f}")

In [ ]:
# Visualise the two groups
df_grp = pd.DataFrame({
    "Score": np.concatenate([group1, group2]),
    "Group": ["Group 1"] * len(group1) + ["Group 2"] * len(group2)
})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=df_grp, x="Group", y="Score", palette="pastel", hue="Group", legend=False,ax=axes[0])
axes[0].set_title("Boxplots by Group")

for grp, color, label in [(group1, "steelblue", "Group 1"),
                           (group2, "coral",     "Group 2")]:
    axes[1].hist(grp, bins=12, alpha=0.5, color=color, label=label, density=True)
axes[1].set_title("Overlaid Histograms")
axes[1].set_xlabel("Score")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.suptitle("Two Independent Samples", fontsize=13)
plt.tight_layout()
plt.show()

---
## 17. Comparing Paired Means

**Paired data** arise when observations come in natural pairs:

- The same subject measured **before and after** a treatment
- Matched subjects (e.g., twins assigned to different conditions)

The key idea: reduce to a **one-sample $t$-test on the differences** $d_i = x_{i,\text{after}} - x_{i,\text{before}}$.

$$t = \frac{\bar{d}}{s_d / \sqrt{n}} \sim t_{n-1}$$

By using paired differences, we **remove subject-to-subject variability**, making the test more powerful than an independent-samples test on the same data.

In [ ]:
# Simulate a before/after experiment (n=30 subjects)
np.random.seed(1)
before = np.random.normal(loc=60, scale=8, size=30)
after  = before + np.random.normal(loc=5, scale=4, size=30)  # true improvement ~5 units

diff = after - before

t_stat, p_val = ttest_rel(before, after)

print(f"Before: mean = {before.mean():.2f},  std = {before.std(ddof=1):.2f}")
print(f"After:  mean = {after.mean():.2f},  std = {after.std(ddof=1):.2f}")
print(f"Differences (after − before): mean = {diff.mean():.2f},  std = {diff.std(ddof=1):.2f}")
print()
print(f"Paired t-test: t = {t_stat:.3f},  p = {p_val:.4f}")
print("(Note: t_stat is negative because SciPy computes before − after internally)")

In [ ]:
# Visualise paired differences
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histogram of differences
axes[0].hist(diff, bins=10, color="teal", edgecolor="white", density=True, alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--", label="No change (0)")
axes[0].axvline(diff.mean(), color="black", linestyle="--",
                label=f"Mean diff = {diff.mean():.2f}")
axes[0].set_title("Distribution of Paired Differences")
axes[0].set_xlabel("After $-$ Before")
axes[0].set_ylabel("Density")
axes[0].legend()

# Paired line plot for first 15 subjects
for i in range(15):
    color = "green" if after[i] > before[i] else "red"
    axes[1].plot(["Before", "After"], [before[i], after[i]],
                 color=color, alpha=0.5, linewidth=1)
axes[1].set_title("Individual Before/After (15 subjects)")
axes[1].set_ylabel("Score")

plt.suptitle("Paired Data Analysis", fontsize=13)
plt.tight_layout()
plt.show()

---
## 18. Comparing Two Population Proportions

To compare proportions from two independent groups, we use a **two-proportion $z$-test**.

**Hypotheses:** $H_0: p_1 = p_2$ vs $H_1: p_1 \neq p_2$ (two-sided)

**Test statistic:**

$$z = \frac{\hat{p}_1 - \hat{p}_2}{\sqrt{\hat{p}(1-\hat{p})\left(\dfrac{1}{n_1} + \dfrac{1}{n_2}\right)}}$$

where the **pooled proportion** is $\hat{p} = \dfrac{x_1 + x_2}{n_1 + n_2}$.

In [ ]:
# Two-proportion z-test
# Group 1: 54 successes out of 120
# Group 2: 30 successes out of 100

n1, x1 = 120, 54
n2, x2 = 100, 30

p1_hat = x1 / n1
p2_hat = x2 / n2
p_pool = (x1 + x2) / (n1 + n2)

se     = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z_stat = (p1_hat - p2_hat) / se
p_val  = 2 * (1 - norm.cdf(abs(z_stat)))   # two-sided

print(f"Group 1: {x1}/{n1} = p̂₁ = {p1_hat:.3f}")
print(f"Group 2: {x2}/{n2} = p̂₂ = {p2_hat:.3f}")
print(f"Pooled proportion (p̂): {p_pool:.3f}")
print(f"Standard error:        {se:.4f}")
print(f"z statistic:           {z_stat:.3f}")
print(f"Two-sided p-value:     {p_val:.4f}")
print()
alpha = 0.05
print("Decision:", "Reject H₀" if p_val < alpha else "Fail to reject H₀")

---
## 19. Contingency Tables and the Chi-Square Test of Independence

A **contingency table** (cross-tabulation) counts joint frequencies for two categorical variables.

The **chi-square test of independence** asks: *are the two variables associated, or are they independent?*

**Hypotheses:** $H_0$: the two variables are independent; $H_1$: they are associated.

**Test statistic:**

$$\chi^2 = \sum_{\text{all cells}} \frac{(O - E)^2}{E}$$

where $O$ = observed count and $E$ = expected count under independence.

The statistic follows a $\chi^2$ distribution with $df = (r-1)(c-1)$ where $r$ and $c$ are the number of rows and columns.

**Rule of thumb:** all expected counts should be $\geq 5$ for the approximation to be reliable.

In [ ]:
# Contingency table: 2 groups × 2 outcomes
table = np.array([[30, 20],
                  [25, 35]])

chi2, p, dof, expected = chi2_contingency(table)

print("Observed counts:")
print(pd.DataFrame(table,
                   columns=["Outcome A", "Outcome B"],
                   index=["Group 1", "Group 2"]))
print()
print("Expected counts (under independence):")
print(pd.DataFrame(expected,
                   columns=["Outcome A", "Outcome B"],
                   index=["Group 1", "Group 2"]).round(2))
print()
print(f"chi-square statistic: {chi2:.3f}")
print(f"degrees of freedom:   {dof}")
print(f"p-value:              {p:.4f}")
print()
print("Decision:", "Reject H₀ (association detected)" if p < 0.05
      else "Fail to reject H₀ (no significant association)")

In [ ]:
# Visualise observed vs expected counts as a heatmap
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

df_obs = pd.DataFrame(table,
                      columns=["Outcome A", "Outcome B"],
                      index=["Group 1", "Group 2"])
df_exp = pd.DataFrame(expected,
                      columns=["Outcome A", "Outcome B"],
                      index=["Group 1", "Group 2"])

sns.heatmap(df_obs, annot=True, fmt="d", cmap="Blues",
            cbar=False, ax=axes[0])
axes[0].set_title("Observed Counts")

sns.heatmap(df_exp, annot=True, fmt=".1f", cmap="Oranges",
            cbar=False, ax=axes[1])
axes[1].set_title("Expected Counts (under $H_0$)")

plt.suptitle("Chi-Square Test of Independence", fontsize=13)
plt.tight_layout()
plt.show()

---
## 20. $F$-Test for Comparing Two Variances

Before using a pooled-variance $t$-test, it is sometimes useful to check whether the population variances are equal.

**Test statistic:**

$$F = \frac{s_1^2}{s_2^2} \sim F(df_1,\, df_2)$$

where $df_1 = n_1 - 1$ and $df_2 = n_2 - 1$.

- $F$ close to 1 → variances are similar (consistent with $H_0: \sigma_1^2 = \sigma_2^2$)
- $F$ far from 1 → evidence against equal variances

> **Note:** The $F$-test for variances is sensitive to non-normality. Levene's test (`scipy.stats.levene`) is a more robust alternative.

In [ ]:
# F-test for comparing two variances
# Group 1: σ ≈ 5 (low variance); Group 2: σ ≈ 8 (higher variance)

np.random.seed(42)
g1 = np.random.normal(loc=50, scale=5, size=40)
g2 = np.random.normal(loc=50, scale=8, size=35)

s1_sq = np.var(g1, ddof=1)
s2_sq = np.var(g2, ddof=1)
df1   = len(g1) - 1
df2   = len(g2) - 1

F_stat = s1_sq / s2_sq
# Two-sided p-value
p_val  = 2 * min(f.cdf(F_stat, df1, df2), 1 - f.cdf(F_stat, df1, df2))

print(f"Group 1: s₁² = {s1_sq:.3f},  df₁ = {df1}")
print(f"Group 2: s₂² = {s2_sq:.3f},  df₂ = {df2}")
print(f"F = s₁²/s₂² = {F_stat:.3f}")
print(f"p-value (two-sided): {p_val:.4f}")
print()
print("Decision:", "Reject H₀ (variances differ significantly)" if p_val < 0.05
      else "Fail to reject H₀ (no significant difference in variances)")

In [ ]:
# Visualise the F-distribution and the observed F statistic

x_range = np.linspace(0, 5, 400)
y_range = f.pdf(x_range, df1, df2)

plt.figure(figsize=(7, 4))
plt.plot(x_range, y_range, color="black", linewidth=2)

# Shade both tails for two-sided test
lower_crit = f.ppf(0.025, df1, df2)
upper_crit = f.ppf(0.975, df1, df2)

plt.fill_between(x_range, y_range, where=(x_range <= lower_crit),
                 color="red", alpha=0.4, label="Rejection region")
plt.fill_between(x_range, y_range, where=(x_range >= upper_crit),
                 color="red", alpha=0.4)
plt.axvline(F_stat, color="blue", linestyle="--",
            label=f"Observed $F = {F_stat:.3f}$")

plt.title(f"$F({df1}, {df2})$ Distribution with Observed $F$ Statistic")
plt.xlabel("$F$")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary

| Scenario | Test | Key statistic |
|---|---|---|
| Two independent means (unequal $\sigma$) | Welch's $t$-test | $t = (\bar{x}_1 - \bar{x}_2)/\sqrt{s_1^2/n_1 + s_2^2/n_2}$ |
| Two independent means (equal $\sigma$) | Pooled $t$-test | $t = (\bar{x}_1 - \bar{x}_2)/(s_p\sqrt{1/n_1+1/n_2})$ |
| Paired observations | Paired $t$-test | $t = \bar{d}/(s_d/\sqrt{n})$ |
| Two proportions | Two-proportion $z$-test | $z = (\hat{p}_1 - \hat{p}_2)/\text{SE}$ |
| Two categorical variables | $\chi^2$ test of independence | $\chi^2 = \sum (O-E)^2/E$ |
| Two variances | $F$-test | $F = s_1^2/s_2^2$ |

**Next:** Part 6 – Regression and ANOVA.